In [1]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd


DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="INDICADORES_ESSALUD"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2= engine2.connect()

from datetime import datetime
from sqlalchemy import text

In [2]:
cas = pd.read_sql_query("SELECT id_cas, cod_cas, cod_red FROM dim_cas WHERE id_cas IS NOT NULL", con=engine2).drop_duplicates(subset=['cod_cas']).dropna()
red = pd.read_sql_query("SELECT id_red, cod_red FROM dim_red", con=engine2)
cas_red = pd.merge(cas, red, how="left", on="cod_red")

In [3]:
cas_red

,id_cas,cod_cas,cod_red,id_red
0,1,830,07,3
1,2,507,13,8
2,3,406,06,2
3,4,118,24,16
4,5,537,99,31
...,...,...,...,...
600,602,A52,95,30
601,603,135,20,12
602,604,802,05,1
603,605,803,99,31


In [4]:
import numpy as np
import oracledb
import pandas as pd
from sqlalchemy import create_engine, text
import sys
from decouple import config

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb

# Configuración de la conexión
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname = config("HOST4_BDI_POSTGRES")
service_name = "WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@', connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

# Tamaño de los bloques
chunk_size = 350000
offset = 0

# Cargar datos de referencia desde PostgreSQL
sexo = pd.read_sql_query("SELECT id_sexo, cod_sex FROM dim_sexo", con=engine2)
tipseg = pd.read_sql_query("SELECT id_tipseg, cod_tse FROM dim_tipseg", con=engine2)
oricas = pd.read_sql_query("SELECT id_oricas, ori_cod FROM dim_oricas", con=engine2)
cas = pd.read_sql_query(f"SELECT id_cas,cod_cas, cod_red FROM  dim_cas where cod_cas is not null", con=connection2)
red = pd.read_sql_query("SELECT id_red, cod_red FROM dim_red", con=engine2)
plansalud = pd.read_sql_query("SELECT id_plansalud, cod_tse, cod_psa FROM dim_plansalud", con=engine2)
tipoparen = pd.read_sql_query("SELECT id_tipoparen, cod_tpa FROM dim_paren", con=engine2)
estreg = pd.read_sql_query("SELECT id_estreg, cod_est FROM dim_estreg", con=engine2)

while True:
    query = f"""
    SELECT *
    FROM (
        SELECT 
            persecnum,
            pernacfec,
            persexocod,
            pertipsegcod,
            peroricenasiadscod,
            percenasiadscod,
            perinsfec,
            pervigfec,
            perfalfec,
            PERPLANSALUCOD,
            pertipoparecod,
            perestregcod,
            ROW_NUMBER() OVER (ORDER BY persecnum) AS rn
        FROM cmper10
    )
    WHERE rn > {offset} AND rn <= {offset + chunk_size}
    """

    chunk = pd.read_sql_query(query, con=engine0)
    if chunk.empty:
        break

    chunk = chunk.rename(columns={
        'persecnum': 'per_sec',
        'pernacfec': 'fec_nac',
        'persexocod': 'cod_sex',
        'pertipsegcod': 'cod_tse',
        'peroricenasiadscod': 'ori_cod',
        'percenasiadscod': 'cod_cas',
        'perinsfec': 'fec_ins',
        'pervigfec': 'fec_vig',
        'perfalfec': 'fec_fal',
        'perplansalucod': 'cod_psa',
        'pertipoparecod': 'cod_tpa',
        'perestregcod': 'cod_est'
    })

    # Transformaciones y combinaciones con otros DataFrames
    sexo['cod_sex'] = sexo['cod_sex'].str.strip()
    chunk['cod_sex'] = chunk['cod_sex'].str.strip()
    chunk = pd.merge(chunk, sexo, on='cod_sex', how="left").drop('cod_sex', axis=1)
    tipseg['cod_tse'] = tipseg['cod_tse'].str.strip()
    chunk['cod_tse'] = chunk['cod_tse'].str.strip()
    plansalud['cod_tse'] = plansalud['cod_tse'].str.strip()
    plansalud['cod_psa'] = plansalud['cod_psa'].str.strip()
    chunk['cod_psa'] = chunk['cod_psa'].str.strip()
    chunk = pd.merge(chunk, plansalud, on=['cod_tse', 'cod_psa'], how="left").drop('cod_psa', axis=1)
    chunk = pd.merge(chunk, tipseg, on='cod_tse', how="left").drop('cod_tse', axis=1)
    oricas['ori_cod'] = oricas['ori_cod'].str.strip()
    chunk['ori_cod'] = chunk['ori_cod'].str.strip()
    chunk = pd.merge(chunk, oricas, on='ori_cod', how="left").drop('ori_cod', axis=1)
    cas['cod_cas'] = cas['cod_cas'].str.strip()
    cas['cod_red'] = cas['cod_red'].str.strip()
    chunk['cod_cas'] = chunk['cod_cas'].str.strip()
    chunk = pd.merge(chunk, cas,how='left',on='cod_cas').drop('cod_cas', axis=1)
    red['cod_red'] = red['cod_red'].str.strip()
    chunk['cod_red'] = chunk['cod_red'].str.strip()
    chunk = pd.merge(chunk, red,how='left',on='cod_red').drop('cod_red', axis=1)
    tipoparen['cod_tpa'] = tipoparen['cod_tpa'].str.strip()
    chunk['cod_tpa'] = chunk['cod_tpa'].str.strip()
    chunk = pd.merge(chunk, tipoparen, on='cod_tpa', how="left").drop('cod_tpa', axis=1)
    estreg['cod_est'] = estreg['cod_est'].str.strip()
    chunk['cod_est'] = chunk['cod_est'].str.strip()
    chunk = pd.merge(chunk, estreg, on='cod_est', how="left").drop('cod_est', axis=1)

    # Convertir columnas de fecha a datetime y manejar NaT
    date_columns = ['fec_nac', 'fec_ins', 'fec_vig', 'fec_fal']

    # Convertir las columnas de fecha a datetime, reemplazando errores con NaT
    for col in date_columns:
        chunk[col] = pd.to_datetime(chunk[col], errors='coerce')

    # Reemplazar NaT por None en columnas de fecha
    for col in date_columns:
        chunk[col] = chunk[col].apply(lambda x: None if pd.isna(x) else x)

    # Convertir el resto de NaN a None
    chunk = chunk.applymap(lambda x: None if pd.isna(x) else x)

    print('subiendo bloques')
    # Upsert en PostgreSQL
    for _, row in chunk.iterrows():
        row_dict = row.to_dict()

        # Verificar y convertir cualquier valor de cadena 'NaT' o pd.NaT a None
        for key, value in row_dict.items():
            if value == 'NaT' or value is pd.NaT or str(value) == 'NaT':
                row_dict[key] = None

        sql = text("""
        INSERT INTO dim_per (per_sec, fec_nac, id_sexo, id_tipseg, id_oricas, id_cas, id_red, fec_ins, fec_vig, fec_fal, id_plansalud, id_tipoparen, id_estreg)
        VALUES (:per_sec, :fec_nac, :id_sexo, :id_tipseg, :id_oricas, :id_cas, :id_red, :fec_ins, :fec_vig, :fec_fal, :id_plansalud, :id_tipoparen, :id_estreg)
        """)
        
        engine2.execute(sql, **row_dict)
    print(f'bloque {_} subido')                                                                                                                                                                                      
    offset += chunk_size

subiendo bloques
bloque 350000 subido
subiendo bloques
bloque 349999 subido
subiendo bloques
bloque 350002 subido
subiendo bloques
bloque 350001 subido
subiendo bloques
bloque 350001 subido
subiendo bloques
bloque 350000 subido
subiendo bloques
bloque 350000 subido
subiendo bloques
bloque 350001 subido
subiendo bloques
bloque 349999 subido
subiendo bloques
bloque 350002 subido
subiendo bloques
bloque 350001 subido
subiendo bloques
bloque 350000 subido
subiendo bloques
bloque 350002 subido
subiendo bloques
bloque 350001 subido
subiendo bloques
bloque 350000 subido
subiendo bloques
bloque 350001 subido
subiendo bloques
bloque 350000 subido
subiendo bloques
bloque 350000 subido
subiendo bloques
bloque 350000 subido
subiendo bloques
bloque 349999 subido
subiendo bloques
bloque 350000 subido
subiendo bloques
bloque 349999 subido
subiendo bloques
bloque 349999 subido
subiendo bloques
bloque 349999 subido
subiendo bloques
bloque 349999 subido
subiendo bloques
bloque 349999 subido
subiendo blo

In [5]:
connection2.close()
connection0.close()
engine2.dispose()
engine0.dispose()